# Mapas do delta de tempo — Porto Alegre

Compara cada cenário com o cenário típico. Valores positivos indicam aumento do tempo de acesso; valores negativos indicam redução.


In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
BASE_DIR = Path.cwd().parents[2]
RESULTS_DIR = BASE_DIR / 'results' / 'porto_alegre'
SCENARIOS = ['inundacao', 'movimento_massa', 'combinado']
LABELS = {'inundacao':'Inundação', 'movimento_massa':'Movimento de massa', 'combinado':'Combinado'}
grid = gpd.read_parquet(RESULTS_DIR / 'prepared' / 'grid.parquet')
def access_time(scenario):
    matrix = pd.read_parquet(RESULTS_DIR / 'matrices' / f'matrix_{scenario}.parquet')
    return matrix.groupby('cell_idx')['travel_time'].mean().rename(scenario)
typical = access_time('typical')
deltas = {scenario: access_time(scenario).subtract(typical) for scenario in SCENARIOS}
for scenario, delta in deltas.items():
    print(f'{scenario:20s} mediana={delta.median():.2f} min | válidos={delta.notna().sum()}')


## Mapas

A escala divergente é comum aos três cenários. Células sem valor permanecem cinzas; isso inclui células sem acesso a POIs.


In [ ]:
valores = pd.concat(deltas.values()).replace([np.inf, -np.inf], np.nan).dropna()
limite = max(1.0, float(np.ceil(max(abs(valores.quantile(0.02)), abs(valores.quantile(0.98))))))
limites = [-limite, -10, -5, -2, 2, 5, 10, limite]
cores = ['#2166ac', '#67a9cf', '#d1e5f0', '#f7f7f7', '#fddbc7', '#ef8a62', '#b2182b']
rotulos = ['< -10 min', '-10 a -5 min', '-5 a -2 min', '-2 a 2 min', '2 a 5 min', '5 a 10 min', '> 10 min']
cmap = ListedColormap(cores)
norm = BoundaryNorm(limites, len(cores))
fig, axes = plt.subplots(1, 3, figsize=(20, 9))
for ax, scenario in zip(axes, SCENARIOS):
    mapa = grid.merge(deltas[scenario].rename('delta_t'), on='cell_idx', how='left')
    mapa.plot(column='delta_t', cmap=cmap, norm=norm, ax=ax, edgecolor='none', missing_kwds={'color':'#bdbdbd'}, legend=False)
    ax.set_title(LABELS[scenario], fontsize=16, fontweight='bold')
    ax.set_axis_off()
fig.legend(handles=[Patch(facecolor=c, edgecolor='#555', label=l) for c, l in zip(cores, rotulos)], loc='lower center', ncol=4, frameon=False, fontsize=12, title='Delta do tempo de acesso em relação ao cenário típico (min)', bbox_to_anchor=(0.5, 0.01))
fig.suptitle('Variação do tempo de acesso por cenário — Porto Alegre', fontsize=20, fontweight='bold')
fig.subplots_adjust(top=0.90, bottom=0.13, wspace=0.04)
plt.show()


## Interpretação

A leitura deve se concentrar na distribuição espacial dos aumentos e reduções do tempo de acesso. Em Porto Alegre, células sem acesso total aparecem separadas das células com variação finita.
